# Shannon-style paragraphs from Google Books n-grams

Each next word is sampled from the conditional distribution of continuations of the
preceding words, taken from the Google Books v3 corpus through the ngrams.dev REST API.

Every cell below is a thin call into the `shannon_ngram` package; none of the logic lives
in this notebook, so the same code runs headless on a server.

## Step 0 — settings

Everything is read from the environment or a `.env` file. Nothing is hard-coded, which is
what lets the same package run on a machine with different paths.

In [ ]:
from shannon_ngram import load_settings, settings_table

s = load_settings()
for name, value in settings_table(s):
    print(f"{name:26} {value}")

## Step 1 — environment check

Which interpreter, which package build, and where the cache will live. No network yet.

In [ ]:
import sys

import shannon_ngram

print("python        ", sys.version.split()[0], "|", sys.executable)
print("shannon_ngram ", shannon_ngram.__version__)
print("cache dir     ", s.cache_path.parent, "| exists:", s.cache_path.parent.exists())

## Step 2 — live API probe

The only cell that requires the network, and the canary for the API changing shape: the
wildcard `*` must come back as a `STAR` query token, and each result must carry exactly one
`inserted` token — the word that filled it.

In [ ]:
from shannon_ngram import NgramClient

probe = NgramClient(s)
page = probe.search("_START_ The *", limit=3)
print([(t.text, t.kind) for t in page.query_tokens])
for n in page.ngrams:
    print(f"  {n.inserted_token.text:12} {n.abs_total_match_count:>14,}")

## Step 3 — client and cache

The same query twice. The second one must cost zero network calls; the cache is on disk, so
it also survives a kernel restart. Empty results are cached too — the backoff walk asks for
the same missing contexts over and over.

In [ ]:
from shannon_ngram import CachedNgramClient, NgramCache

client = CachedNgramClient(probe, NgramCache(s.cache_path))
client.search_all("on the table *")
before = client.network_calls
client.search_all("on the table *")
print("network calls for the repeat:", client.network_calls - before, "|", client.cache.stats())

## Step 4 — token filter and case aggregation

The API returns case variants as separate rows (`on the table` and `on the Table` both
match), so counts are merged by the casefolded next word. `flags=cs` would instead *drop*
rows, discarding real probability mass — hence aggregation, not the flag.

In [ ]:
from shannon_ngram.model import aggregate_counts, is_usable_token

rows = client.search_all("on the table *")
print("rows:", len(rows), "| usable:", sum(is_usable_token(n.inserted_token) for n in rows))
merged = aggregate_counts(rows, case_sensitive=False)
strict = aggregate_counts(rows, case_sensitive=True)
print("merged entries:", len(merged), "| kept apart by case:", len(strict))

## Step 5 — counts to probabilities

Divide by the total. Entropy in bits says how undecided the corpus is about what comes next:
0 bits would be certainty, `log2(candidates)` would be a coin toss between all of them.

In [ ]:
from shannon_ngram.model import counts_to_distribution, entropy_bits

dist = counts_to_distribution(merged)
for word, p in sorted(dist.items(), key=lambda kv: -kv[1])[:15]:
    print(f"  {word:14} {merged[word]:>14,}  p={p:.4f}")
print("sum(p) =", sum(dist.values()), "| entropy =", round(entropy_bits(dist), 3), "bits")

## Step 6 — backoff

A four-word context that the corpus has never seen returns nothing, so the model drops the
leftmost word and asks again, down to a single word. The level actually used is recorded on
every step.

In [ ]:
from shannon_ngram import NextWordModel

model = NextWordModel(client, s)
rare = ["a", "small", "green", "quixotic"]
print("levels to try:", model.levels(rare, at_sentence_start=False))
d = model.distribution(rare)
print("level used:", d.level, "| context:", d.context, "| candidates:", d.candidates)

## Step 7 — the sampler

Ten draws from the same context. Candidates are sorted `(-count, word)` before the
cumulative sum, so the seed alone decides the output — dict order never leaks in.

In [ ]:
from shannon_ngram import ParagraphGenerator

gen = ParagraphGenerator(model, s, seed=42)
for i in range(10):
    word, step = gen.sample_next(["on", "the", "table"], index=i)
    print(f"  {word:14} p={step.probability:.4f} level={step.level} backed_off={step.backed_off}")

## Step 8 — one sentence

A sentence starts at `_START_` and ends at `_END_`, at the 40-word length guard, or at a
dead end (backoff exhausted at one word). Whichever happened is recorded in `stop_reason`.

In [ ]:
sentence = gen.sentence()
print(sentence.text)
print("stop_reason:", sentence.stop_reason, "| words sampled:", len(sentence.steps))

## Step 9 — a paragraph

At most `NGRAM_MAX_SENTENCES` sentences. On a cold cache this is the slow cell: every word
is a fresh HTTP request, and the batch endpoint cannot help (it reads `*` as a literal).

In [ ]:
paragraph = gen.paragraph()
print(paragraph.text)
print()
print("sentences:", len(paragraph.sentences), "<=", s.max_sentences)
print("stop reasons:", [x.stop_reason for x in paragraph.sentences])
assert len(paragraph.sentences) <= s.max_sentences

## Step 10 — diagnostics

One row per sampled word, then entropy per step. Marked points are steps where the full
context found nothing and the model had to back off — those are the uncertain moments.

In [ ]:
from shannon_ngram import diagnostics

for row in diagnostics.step_table(paragraph.steps)[:12]:
    print(row)
print("mean entropy:", round(diagnostics.mean_entropy(paragraph.steps), 3), "bits")

In [ ]:
ax = diagnostics.plot_entropy(paragraph.steps)
ax.figure

## Step 11 — Shannon-style comparison, context length 1 to 4

Shannon's point: the longer the context, the more English-like the output. Same seed for all
four, so the difference is the context length and nothing else. `effective_level` is the mean
context length actually used — backoff means it can be lower than requested.

In [ ]:
from dataclasses import replace

lengths = (1, 2, 3, 4)
paras = diagnostics.compare_context_lengths(replace(s, max_sentences=2), client, 42, lengths)
for row in diagnostics.comparison_table(lengths, paras):
    print(row)

In [ ]:
for length, para in zip(lengths, paras):
    print(f"--- context length {length} ---")
    print(para.text)
    print()

## Step 12 — credits

Data: Google Books Ngram v3 (CC BY 3.0) via ngrams.dev

Served by the [ngrams.dev](https://ngrams.dev) REST API. The English corpus drops n-grams
seen fewer than 40 times, so the tail of every distribution here is truncated at the source.